In [18]:
from minio import Minio
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, when, length 
from pyspark.sql.types import NumericType, IntegerType, LongType, FloatType, DoubleType, DecimalType, DateType, TimestampType
from pyspark.sql import Window
import pyspark.sql.functions as F
from dotenv import load_dotenv, find_dotenv
import psycopg2
load_dotenv(find_dotenv())
import os

JAR_PATH_1 = os.path.abspath("./jars/hadoop-aws-3.4.0.jar")
JAR_PATH_2 = os.path.abspath("./jars/aws-sdk-s3-2.29.52.jar")

JARS_LIST = f"{JAR_PATH_1},{JAR_PATH_2}"

spark = (
    SparkSession.builder.appName("analysis")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.jars.repositories", "https://repo1.maven.org/maven2/")
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY"))
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

In [19]:
spark = (
    SparkSession.builder.appName("analysis")
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY"))
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
)

In [20]:
spark

In [21]:
conn = psycopg2.connect(
    host="localhost",
    database=os.getenv("POSTGRES_DATABASE_NAME"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
)


LOAD DATA FROM POSTGRE DATABASE REMAINING

In [22]:
import os
import psycopg2
from pyspark.sql.types import StructType, StructField, StringType


DB_HOST = "localhost" 
DB_PORT = "5432" 
DB_NAME = os.getenv("POSTGRES_DATABASE_NAME", "pulse")
DB_USER = os.getenv("POSTGRES_USER", "postgres")
DB_PASS = os.getenv("POSTGRES_PASSWORD", "postgres")

def get_agg_tables():
    try:
        print(f"Connecting to Postgres at {DB_HOST}:{DB_PORT}...")
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USER,
            password=DB_PASS
        )
        cursor = conn.cursor()

        cursor.execute("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public' 
            AND table_name LIKE 'agg_%'
        """)
        
        tables = [row[0] for row in cursor.fetchall()]
        print(f"Found tables: {tables}")

        spark_dfs = {}

        jdbc_url = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
        connection_properties = {
            "user": DB_USER,
            "password": DB_PASS,
            "driver": "org.postgresql.Driver"
        }

        for table in tables:
            print(f"Processing table: {table}...")
            df = spark.read.jdbc(url=jdbc_url, table=f'"{table}"', properties=connection_properties)
            spark_dfs[table]= df

        cursor.close()
        conn.close()
        return spark_dfs

    except Exception as e:
        print(f"Error: {e}")
        # Print full stack trace for debugging if needed
        import traceback
        traceback.print_exc()
        return {}

dataframes = get_agg_tables()

Connecting to Postgres at localhost:5432...
Found tables: ['agg_customer_sessions', 'agg_customers', 'agg_inventory', 'agg_marketing_campaigns', 'agg_order_items', 'agg_orders', 'agg_payments', 'agg_products', 'agg_reviews', 'agg_shopping_cart', 'agg_suppliers', 'agg_wishlist', 'agg_categories', 'agg_daily_aggregations', 'agg_weekly_aggregations', 'agg_monthly_aggregations', 'agg_country_aggregations', 'agg_state_aggregations', 'agg_city_aggregations', 'agg_cart_abandonment_analysis', 'agg_product_inventory_health', 'agg_supplier_inventory_health', 'agg_rfm_segmentation', 'agg_rfm_segment_summary', 'agg_product_affinity', 'agg_top_product_pairs', 'agg_product_recommendations', 'agg_category_affinity', 'agg_global_aggregations']
Processing table: agg_customer_sessions...
Processing table: agg_customers...
Processing table: agg_inventory...
Processing table: agg_marketing_campaigns...
Processing table: agg_order_items...
Processing table: agg_orders...
Processing table: agg_payments...
P

In [23]:
dataframes["agg_rfm_segment_summary"].show(5)

+----------------------+--------------+------------------+------------------+--------------------+------------------+
|customer_segment_label|customer_count|       avg_revenue|        avg_orders|avg_days_since_order|     avg_rfm_score|
+----------------------+--------------+------------------+------------------+--------------------+------------------+
|                Others|           155| 632.7769677419357|1.1096774193548387|  1585.5612903225806|2.1849462365591403|
|       Loyal Customers|           147|2285.6266666666666| 1.945578231292517|  195.69387755102042| 3.902494331065758|
|             Champions|           133|3171.4133082706776| 2.654135338345865|   97.00751879699249| 4.706766917293235|
|               At Risk|           132| 2211.483030303031| 1.696969696969697|  480.43939393939394|3.0909090909090926|
|         New Customers|            81|  790.341975308642|1.3209876543209877|  105.71604938271605|3.1975308641975295|
+----------------------+--------------+-----------------

ALL NULL ROWs AND DATAFRAME CHECK 

In [24]:
def is_column_all_null_or_zero(df, col_name):
    if df is None:
        return True                    

    if col_name not in df.columns:
        return True                   

    col_type = dict(df.dtypes)[col_name]
    non_null_count = df.agg(
        F.count(F.col(col_name)).alias("non_null_count")
    ).collect()[0]["non_null_count"]
    if non_null_count == 0:
        return True                   

    
    if col_type in ("int", "bigint", "double", "float", "decimal", "smallint", "tinyint"):
        non_zero_non_null_count = df.agg(
            F.sum(
                F.when(
                    (F.col(col_name).isNotNull()) & (F.col(col_name) != 0), 1
                ).otherwise(0)
            ).alias("non_zero_non_null_count")
        ).collect()[0]["non_zero_non_null_count"]

        if non_zero_non_null_count == 0:
            return True                 

    return False

Time Grain function

In [25]:
def add_time_grain(df, date_col, grain="day"):
    if grain == "day":
        return df.withColumn("grain_date", F.col(date_col))
    elif grain == "week":
        return df.withColumn("grain_year", F.year(date_col)) \
                 .withColumn("grain_week", F.weekofyear(date_col))
    elif grain == "month":
        return df.withColumn("grain_year", F.year(date_col)) \
                 .withColumn("grain_month", F.month(date_col))
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

A dataframe to contain all the analysis results to keep track easily 

In [26]:
analysis = {}

# Customer Related Analysis 

Adding Date Column

In [27]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    dataframes["agg_customers"] = dataframes["agg_customers"].withColumn(
    "account_created_date",
    F.to_date("account_created_at")
    )

In [28]:
if not is_column_all_null_or_zero(dataframes["agg_orders"], "order_placed_at"):
    dataframes["agg_orders"] = dataframes["agg_orders"].withColumn(
    "order_date",
    F.to_date("order_placed_at")
    )

“total units sold” per order

In [29]:
order_units = (
    dataframes["agg_order_items"]
    .groupBy("order_id")
    .agg(F.sum("quantity").alias("units_sold"))
)

dataframes["agg_orders"] = (
    dataframes["agg_orders"]
    .join(order_units, on="order_id", how="left")
    .fillna({"units_sold": 0})
)

Core KPIs per period

In [30]:
def core_kpis_over_time(df,date_col, grain="day"):
    df_g = add_time_grain(df, date_col="order_placed_at", grain=grain)

    if grain == "day":
        group_cols = ["grain_date"]
        order_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
        order_cols = ["grain_year", "grain_week"]
    else:  # "month"
        group_cols = ["grain_year", "grain_month"]
        order_cols = ["grain_year", "grain_month"]

    kpi = (
        df_g.groupBy(*group_cols)
            .agg(
                F.countDistinct("order_id").alias("total_orders"),
                F.sum("units_sold").alias("total_units_sold"),
                F.sum("total_amount").alias("total_revenue"),
                F.sum("order_profit").alias("gross_profit"),
                F.sum("net_profit").alias("net_profit")
            ).fillna({
                "total_units_sold": 0, 
                "total_revenue": 0, 
                "gross_profit": 0, 
                "net_profit": 0
            })
            .withColumn(
                "aov",
                F.when(F.col("total_orders") > 0,
                       F.col("total_revenue") / F.col("total_orders"))
                 .otherwise(F.lit(0.0))
            )
            .withColumn(
                "margin_pct",
                F.when(F.col("total_revenue") > 0,
                       F.col("gross_profit") / F.col("total_revenue"))
                 .otherwise(F.lit(0.0))
            )
            .orderBy(*order_cols)
    )

    return kpi
if not is_column_all_null_or_zero(dataframes["agg_orders"], "order_placed_at") and not is_column_all_null_or_zero(dataframes["agg_orders"], "units_sold") and not is_column_all_null_or_zero(dataframes["agg_orders"], "total_amount") and not is_column_all_null_or_zero(dataframes["agg_orders"], "order_profit") and not is_column_all_null_or_zero(dataframes["agg_orders"], "net_profit"):
    analysis["business_health_daily"]= core_kpis_over_time(dataframes["agg_orders"],"order_placed_at", grain="day")
    analysis["business_health_weekly"] = core_kpis_over_time(dataframes["agg_orders"],"order_placed_at", grain="week")
    analysis["business_health_monthly"] = core_kpis_over_time(dataframes["agg_orders"],"order_placed_at", grain="month")
else:
    print("One or more required columns are all null or zero in agg_orders dataframe.")

In [33]:
analysis["business_health_monthly"].show(3)

+----------+-----------+------------+----------------+------------------+-------------------+-------------------+------------------+-------------------+
|grain_year|grain_month|total_orders|total_units_sold|     total_revenue|       gross_profit|         net_profit|               aov|         margin_pct|
+----------+-----------+------------+----------------+------------------+-------------------+-------------------+------------------+-------------------+
|      2023|         11|          32|            2778|33223.049999999996|-445099.42999999993|         -261524.89|1038.2203124999999|-13.397307893164534|
|      2023|         12|          85|            6780|100319.92000000007| -832880.0200000001|         -611248.91|1180.2343529411773| -8.302239674832272|
|      2024|          1|          75|            5798|          75114.15| -821419.3000000002|-481553.85000000003|1001.5219999999999|-10.935613329845312|
+----------+-----------+------------+----------------+------------------+---------

Analysis: Net Revenue vs. Net Profit financial health over time 

In [ ]:
def analyze_financial_health(orders_df, date_col="order_placed_at", grain="month"):
    df_g = add_time_grain(orders_df, date_col, grain)
    
    if grain == "day":
        group_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
    else: # month
        group_cols = ["grain_year", "grain_month"]

    financial_df = (
        df_g.groupBy(*group_cols)
            .agg(
                F.sum("net_revenue").alias("total_net_revenue"),
                F.sum("net_profit").alias("total_net_profit"),
                F.count("order_id").alias("total_orders")
            ).fillna({
                "total_net_revenue": 0,
                "total_net_profit": 0,
                "total_orders": 0
            })
            .withColumn("period_margin_pct", 
                        F.round((F.col("total_net_profit") / F.col("total_net_revenue")) * 100, 2))
            .orderBy(*group_cols)
    )
    
    return financial_df
if not is_column_all_null_or_zero(dataframes["agg_orders"], "order_placed_at"):
    analysis["financial_health_daily"] = analyze_financial_health(dataframes["agg_orders"],"order_placed_at", grain="day")
    analysis["financial_health_monthly"] = analyze_financial_health(dataframes["agg_orders"], "order_placed_at", grain="month")

Analysis: Margin by Category (The "Drag" Analysis)

In [35]:
def analyze_category_margins(products_df):
   
    category_df = (
        products_df.groupBy("category")
            .agg(
                F.avg("profit_margin").alias("avg_profit_margin"),
                F.sum("total_profit").alias("total_category_profit"),
                F.sum("total_revenue").alias("total_category_revenue"),
                F.sum("total_units_sold").alias("units_sold")
            ).fillna({
                "avg_profit_margin": 0,
                "total_category_profit": 0,
                "total_category_revenue": 0,
                "units_sold": 0
            })
            .orderBy(F.col("avg_profit_margin").asc())
    )
    
    return category_df

if not is_column_all_null_or_zero(dataframes["agg_products"], "category"):
    analysis["low_margin_categories"] = analyze_category_margins(dataframes["agg_products"])
else: 
    print("Category column is all NULL or zero; skipping category margin analysis.")

active customers over time

In [41]:
def active_customers_over_time(df,date_col,grain="day"):
    # Ensure we have a proper date column

    # Apply common time-grain helper
    df_g = add_time_grain(df, date_col="account_created_date", grain=grain)

    # Decide group/order columns based on grain
    if grain == "day":
        group_cols = ["grain_date"]
        order_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
        order_cols = ["grain_year", "grain_week"]
    elif grain == "month":
        group_cols = ["grain_year", "grain_month"]
        order_cols = ["grain_year", "grain_month"]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df_g.filter(F.col("is_active") == True)
            .groupBy(*group_cols)
            .agg(F.countDistinct("customer_id").alias("active_customers"))
            .fillna({"active_customers": 0})
            .orderBy(*order_cols)
    )
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_date"):
    analysis["active_customers_daily"]   = active_customers_over_time(dataframes["agg_customers"], "day")
    analysis["active_customers_weekly"]  = active_customers_over_time(dataframes["agg_customers"], "week")
    analysis["active_customers_monthly"] = active_customers_over_time(dataframes["agg_customers"], "month")
else:
    print("Account created at column is all NULL or zero; skipping active customers over time analysis.")

In [45]:
analysis["active_customers_monthly"].show(3)

+----------+----------------+
|grain_date|active_customers|
+----------+----------------+
|1900-01-01|               5|
|1952-12-15|               1|
|2020-11-19|               1|
+----------+----------------+
only showing top 3 rows



account_status over time 

In [47]:
def status_distribution_over_time(df,date_col, grain="day"):

    # Apply shared time-grain helper
    df_g = add_time_grain(df,  date_col="account_created_date", grain=grain)

    # Group/order columns based on grain
    if grain == "day":
        group_cols = ["grain_date"]
        order_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
        order_cols = ["grain_year", "grain_week"]
    elif grain == "month":
        group_cols = ["grain_year", "grain_month"]
        order_cols = ["grain_year", "grain_month"]
    else:
        raise ValueError("grain must be 'day', 'week', or 'month'")

    return (
        df_g.groupBy(*(group_cols + ["account_status"]))
            .agg(F.countDistinct("customer_id").alias("customer_count"))
            .fillna({"customer_count": 0})
            .orderBy(*order_cols, "account_status")
    )
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    analysis["customer_account_status_distribution_daily"]   = status_distribution_over_time(dataframes["agg_customers"], "day")
    analysis["customer_account_status_distribution_weekly"]  = status_distribution_over_time(dataframes["agg_customers"], "week")
    analysis["customer_account_status_distribution_monthly"] = status_distribution_over_time(dataframes["agg_customers"], "month")
else:
    print("Account created at column is all NULL or zero; skipping status distribution over time analysis.")

New customers per day/week/month

In [48]:
def new_customers(df, date_col="account_created_date", grain="day"):
    df_g = add_time_grain(df, date_col=date_col, grain=grain)

    if grain == "day":
        group_cols = ["grain_date"]
        order_cols = ["grain_date"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week"]
        order_cols = ["grain_year", "grain_week"]
    else:   # month
        group_cols = ["grain_year", "grain_month"]
        order_cols = ["grain_year", "grain_month"]

    new_df = (
        df_g.groupBy(*group_cols)
            .agg(F.countDistinct("customer_id").alias("new_customers"))
            .fillna({"new_customers": 0})
            .orderBy(*order_cols)
    )
    return new_df
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_date"):
    analysis["new_customers_daily"] = new_customers(dataframes["agg_customers"],"account_created_date", "day")
    analysis["new_customers_weekly"]  = new_customers(dataframes["agg_customers"], "account_created_date", "week")
    analysis["new_customers_monthly"]= new_customers(dataframes["agg_customers"], "account_created_date", "month")
else:
    print("Account created at column is all NULL or zero; skipping new customers analysis.")

Cumulative customer growth curve

In [49]:
def cumulative_customers(df, date_col="account_created_at", grain="day"):
    new_df = new_customers(df, date_col, grain)

    # Define window by time order
    if grain == "day":
        window = Window.orderBy("grain_date") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    elif grain == "week":
        window = Window.orderBy("grain_year", "grain_week") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    else:   # month
        window = Window.orderBy("grain_year", "grain_month") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)

    cum_df = new_df.withColumn(
        "cumulative_customers",
        F.sum("new_customers").over(window)
    ).fillna({"cumulative_customers": 0})   
    
    return cum_df
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_date"):
    analysis["cumulative_customers_daily"]   = cumulative_customers(dataframes["agg_customers"],"account_created_date", "day")
    analysis["cumulative_customers_weekly"]  = cumulative_customers(dataframes["agg_customers"], "account_created_date", "week")
    analysis["cumulative_customers_monthly"] = cumulative_customers(dataframes["agg_customers"], "account_created_date", "month")
else:
    print("Account created at column is all NULL or zero; skipping cumulative customers analysis.")

Total new customers by geography + time

In [50]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at"):
    geo_acquisition = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province", "city")
        .agg(F.countDistinct("customer_id").alias("new_customers"))
    )
else:
    print("Account created at column is all NULL or zero; skipping geo acquisition analysis.")

def geo_acquisition_over_time(df, date_col="account_created_at", grain="day"):
    df_g = add_time_grain(df, date_col=date_col, grain=grain)

    if grain == "day":
        group_cols = ["grain_date", "country", "state_province", "city"]
        order_cols = ["grain_date", "country", "state_province", "city"]
    elif grain == "week":
        group_cols = ["grain_year", "grain_week", "country", "state_province", "city"]
        order_cols = ["grain_year", "grain_week", "country", "state_province", "city"]
    else:  # month
        group_cols = ["grain_year", "grain_month", "country", "state_province", "city"]
        order_cols = ["grain_year", "grain_month", "country", "state_province", "city"]

    return (
        df_g.groupBy(*group_cols)
            .agg(F.countDistinct("customer_id").alias("new_customers"))
            .fillna({"new_customers": 0})
            .orderBy(*order_cols)
    )

if not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_date"):
    analysis["new_customers_geo_acquisition_daily"]   = geo_acquisition_over_time(dataframes["agg_customers"], "account_created_date", "day")
    analysis["new_customers_geo_acquisition_monthly"] = geo_acquisition_over_time(dataframes["agg_customers"], "account_created_date", "month")
else:
    print("Account created at column is all NULL or zero; skipping geo acquisition over time analysis.")

Customer distribution by age group, city, state, country

In [51]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_age_group"):
    analysis["customer_age_group_distribution"] = (
        dataframes["agg_customers"]
        .groupBy("customer_age_group")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .fillna({"customer_count": 0})
        .orderBy("customer_age_group")
    )
else: 
    print("Customer age group column is all NULL or zero; skipping age group distribution analysis.")
    
if not is_column_all_null_or_zero(dataframes["agg_customers"], "country") and not is_column_all_null_or_zero(dataframes["agg_customers"], "state_province") and not is_column_all_null_or_zero(dataframes["agg_customers"], "city"):
    analysis["customer_city_distribution"] = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province", "city")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .fillna({"customer_count": 0})
        .orderBy("country", "state_province", "city")
)
else:
    print("Country column is all NULL or zero; skipping city distribution analysis.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "country") and not is_column_all_null_or_zero(dataframes["agg_customers"], "state_province"):
    analysis["customer_state_distribution"] = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .fillna({"customer_count": 0})
        .orderBy("country", "state_province")
)
    
else:
    print("Country or state_province column is all NULL or zero; skipping state distribution analysis.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "country"):
    analysis["customer_country_distribution"] = (
        dataframes["agg_customers"]
        .groupBy("country")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .fillna({"customer_count": 0})
    .orderBy("country")
)
else:
    print("Country column is all NULL or zero; skipping country distribution analysis.")

# Age group distribution and spending patterns

In [52]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_age_group") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_total_spent") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue"):
    analysis["customer_age_group_spending"] = (
        dataframes["agg_customers"]
        .groupBy("customer_age_group")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("order_total_spent").alias("avg_order_total_spent"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.sum("order_total_spent").alias("total_spent"),
            F.sum("total_revenue").alias("total_revenue_age_group")
        ).fillna({"avg_order_total_spent": 0,
                "avg_clv": 0,
                "total_spent": 0,
                "total_revenue_age_group": 0})
        .orderBy("customer_age_group")
    )
else: 
    print("Customer age group column is all NULL or zero; skipping age group spending analysis.")

Gender-based product preferences

In [53]:
if not is_column_all_null_or_zero(dataframes["agg_orders"], "order_id") and not is_column_all_null_or_zero(dataframes["agg_orders"], "customer_id"):
    cust_orders = (
        dataframes["agg_orders"]
        .select("order_id", "customer_id")
        .join(
            dataframes["agg_customers"].select("customer_id", "gender"),
            on="customer_id",
            how="inner"
        )
    )
else:
    print("Order ID or Customer ID column is all NULL or zero; skipping customer orders join.")

if not is_column_all_null_or_zero(dataframes["agg_orders"], "order_id") and not is_column_all_null_or_zero(dataframes["agg_order_items"], "order_id") and not is_column_all_null_or_zero(dataframes["agg_order_items"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_order_items"], "quantity") and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_products"], "product_name") and not is_column_all_null_or_zero(dataframes["agg_products"], "category") and not is_column_all_null_or_zero(dataframes["agg_products"], "sub_category") and not is_column_all_null_or_zero(dataframes["agg_products"], "brand"):
    cust_order_items = (
        cust_orders
        .join(dataframes["agg_order_items"].select("order_id", "product_id", "quantity"), on="order_id", how="inner")
        .join(dataframes["agg_products"].select("product_id", "product_name", "category", "sub_category", "brand"),
            on="product_id",
            how="left")
    )
else:
    print("One of the required columns in orders, order items, or products is all NULL or zero; skipping customer order items join.")
# Gender-based preferences by category
if not is_column_all_null_or_zero(cust_order_items, "gender") and not is_column_all_null_or_zero(cust_order_items, "category") and not is_column_all_null_or_zero(cust_order_items, "quantity") and not is_column_all_null_or_zero(cust_order_items, "product_id") and not is_column_all_null_or_zero(cust_order_items, "order_id"):    
    analysis["gender_category_preference"] = (
        cust_order_items
        .groupBy("gender", "category")
        .agg(
            F.sum("quantity").alias("total_units"),
            F.countDistinct("product_id").alias("distinct_products"),
            F.countDistinct("order_id").alias("orders_count")
        )
        .fillna({"total_units": 0, "distinct_products": 0, "orders_count": 0})
        .orderBy("gender", F.col("total_units").desc())
    )
else:
    print("One of the required columns in customer order items is all NULL or zero; skipping gender category preference analysis.")

if not is_column_all_null_or_zero(cust_order_items, "gender") and not is_column_all_null_or_zero(cust_order_items, "product_id") and not is_column_all_null_or_zero(cust_order_items, "product_name") and not is_column_all_null_or_zero(cust_order_items, "category") and not is_column_all_null_or_zero(cust_order_items, "quantity") and not is_column_all_null_or_zero(cust_order_items, "order_id"):
    analysis["gender_product_preference"] = (
        cust_order_items
        .groupBy("gender", "product_id", "product_name", "category")
        .agg(
            F.sum("quantity").alias("total_units"),
            F.countDistinct("order_id").alias("orders_count")
        ).fillna({"total_units": 0, "orders_count": 0})
        .orderBy("gender", F.col("total_units").desc())
    )
else:
    print("One of the required columns in customer order items is all NULL or zero; skipping gender product preference analysis.")

In [54]:
analysis["gender_product_preference"].show(3)

+------+----------+--------------------+-------------+-----------+------------+
|gender|product_id|        product_name|     category|total_units|orders_count|
+------+----------+--------------------+-------------+-----------+------------+
|Female|      1012|       Amazon Tablet|Winter Sports|       1032|          10|
|Female|      1014|Apple Smartwatch ...|     Dressers|        876|          15|
|Female|      1007|Microsoft Speaker...|     Consoles|        722|          10|
+------+----------+--------------------+-------------+-----------+------------+
only showing top 3 rows



New vs returning customers

In [55]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "is_repeat_customer"):
    dataframes["agg_customers"] = dataframes["agg_customers"].withColumn(
        "customer_type",
        F.when(F.col("is_repeat_customer") == 1, F.lit("returning"))
        .otherwise(F.lit("new"))
    )
else:
    print("is_repeat_customer column is all NULL or zero; skipping customer type classification.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_type") and not is_column_all_null_or_zero(dataframes["agg_customers"], "country"):
    analysis["new_vs_returning_customer_country"] = (
        dataframes["agg_customers"]
        .groupBy("country", "customer_type")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .fillna({"customer_count": 0})
        .orderBy("country", "customer_type")
    )
else:
    print("Country or customer_type column is all NULL or zero; skipping new vs returning country analysis.")
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_type") and not is_column_all_null_or_zero(dataframes["agg_customers"], "country") and not is_column_all_null_or_zero(dataframes["agg_customers"], "state_province") and not is_column_all_null_or_zero(dataframes["agg_customers"], "city"):
    analysis["new_vs_returning_customer_city"] = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province", "city", "customer_type")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .orderBy("country", "state_province", "city", "customer_type")
    )
else:
    print("One of the required columns (customer_type, country, state_province, city) is all NULL or zero; skipping new vs returning city analysis.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_type") and not is_column_all_null_or_zero(dataframes["agg_customers"], "country") and not is_column_all_null_or_zero(dataframes["agg_customers"], "state_province"):
    new_vs_returning_state = (
        dataframes["agg_customers"]
        .groupBy("country", "state_province", "customer_type")
    .agg(F.countDistinct("customer_id").alias("customer_count"))
    .fillna({"customer_count": 0})
    .orderBy("country", "state_province", "customer_type")
)
else:
    print("One of the required columns (customer_type, country, state_province) is all NULL or zero; skipping new vs returning state analysis.")

Total & average engagement per customer

In [56]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_sessions") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_pages_viewed") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_products_viewed"):
    analysis["customer_engagement"] = dataframes["agg_customers"].select(
        "customer_id",
        "total_sessions",
        "total_pages_viewed",
        "total_products_viewed"
    )

    analysis["customer_engagement_summary"] = dataframes["agg_customers"].agg(
        F.sum("total_sessions").alias("total_sessions_all_customers"),
        F.avg("total_sessions").alias("avg_sessions_per_customer"),
        F.sum("total_pages_viewed").alias("total_pages_viewed_all_customers"),
        F.avg("total_pages_viewed").alias("avg_pages_viewed_per_customer"),
        F.sum("total_products_viewed").alias("total_products_viewed_all_customers"),
        F.avg("total_products_viewed").alias("avg_products_viewed_per_customer")
    ).fillna({"total_sessions_all_customers": 0,
            "avg_sessions_per_customer": 0,
            "total_pages_viewed_all_customers": 0,
            "avg_pages_viewed_per_customer": 0,
            "total_products_viewed_all_customers": 0,
            "avg_products_viewed_per_customer": 0
            })
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping engagement analysis.")

Session-to-order behavior

In [57]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "session_conversion_rate") and not is_column_all_null_or_zero(dataframes["agg_customers"], "cart_abandonment_rate"):
    analysis["session_to_order_analysis"] = dataframes["agg_customers"].agg(
        F.avg("session_conversion_rate").alias("avg_session_conversion_rate"),
        F.avg("cart_abandonment_rate").alias("avg_cart_abandonment_rate")
    ).fillna({
        "avg_session_conversion_rate": 0,
        "avg_cart_abandonment_rate": 0
    })
else:
    print("One of the required columns (session_conversion_rate, cart_abandonment_rate) is all NULL or zero; skipping session to order analysis.")

Top customers based on revenue

In [58]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_segment") and not is_column_all_null_or_zero(dataframes["agg_customers"], "rfm_segment"):
    analysis["top_customers_by_revenue"] = (
        dataframes["agg_customers"]
        .fillna({"total_revenue": 0.0})
        .select("customer_id", "total_revenue", "customer_lifetime_value", "customer_segment", "rfm_segment")
        .orderBy(F.col("total_revenue").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping top customers by revenue analysis.")

One of the required columns in agg_customers is all NULL or zero; skipping top customers by revenue analysis.


top customers based on Profit

In [59]:
if not is_column_all_null_or_zero(dataframes["agg_orders"], "order_profit") and not is_column_all_null_or_zero(dataframes["agg_orders"], "net_profit") and not is_column_all_null_or_zero(dataframes["agg_orders"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_orders"], "order_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_segment") and not is_column_all_null_or_zero(dataframes["agg_customers"], "rfm_segment") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value"):
    customer_profit = (
        dataframes["agg_orders"]
        .groupBy("customer_id")
        .agg(
            F.sum("order_profit").alias("total_order_profit"),
            F.sum("net_profit").alias("total_net_profit"),
            F.countDistinct("order_id").alias("orders_count")
        )
    )


    customer_profit_enriched = (
        customer_profit.alias("p")
        .join(
            dataframes["agg_customers"].select("customer_id", "customer_segment", "rfm_segment", "total_revenue", "customer_lifetime_value").alias("c"),
            on="customer_id",
            how="left"
        )
    )

    analysis["top_customers_by_profit"] = (
        customer_profit_enriched
        .orderBy(F.col("total_net_profit").desc_nulls_last())
    )

else:
    print("One of the required columns in agg_orders or agg_customers is all NULL or zero; skipping top customers by profit analysis.")

One of the required columns in agg_orders or agg_customers is all NULL or zero; skipping top customers by profit analysis.


distribution percentage for conversion & abandonment

In [60]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "session_conversion_rate"):
    conv_percentage = dataframes["agg_customers"].withColumn(
        "session_conversion_percentage",
        F.when(F.col("session_conversion_rate") < 0.1, "<10%")
        .when(F.col("session_conversion_rate") < 0.25, "10–25%")
        .when(F.col("session_conversion_rate") < 0.5, "25–50%")
        .when(F.col("session_conversion_rate") < 0.75, "50–75%")
        .otherwise("75%+")
    )
    analysis["session_conversion_distribution"] = (
        conv_percentage
        .groupBy("session_conversion_percentage")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .fillna({"customer_count": 0})
        .orderBy("session_conversion_percentage")
    )
else:
    print("session_conversion_rate column is all NULL or zero; skipping session conversion percentage calculation.")

if not is_column_all_null_or_zero(dataframes["agg_customers"], "cart_abandonment_rate"):
    abandon_percentage = dataframes["agg_customers"].withColumn(
        "cart_abandonment_percentage",
        F.when(F.col("cart_abandonment_rate") < 0.1, "<10%")
        .when(F.col("cart_abandonment_rate") < 0.25, "10–25%")
        .when(F.col("cart_abandonment_rate") < 0.5, "25–50%")
        .when(F.col("cart_abandonment_rate") < 0.75, "50–75%")
        .otherwise("75%+")
    )

    analysis["cart_abandonment_distribution"] = (
        abandon_percentage
        .groupBy("cart_abandonment_percentage")
        .agg(F.countDistinct("customer_id").alias("customer_count"))
        .fillna({"customer_count": 0})
        .orderBy("cart_abandonment_percentage")
    )
else:
    print("cart_abandonment_rate column is all NULL or zero; skipping cart abandonment percentage calculation.")

Basic correlation: tenure vs. spending

In [61]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_tenure_days") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_total_spent") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value"):
    analysis["customer_tenure_correlations"] = dataframes["agg_customers"].select(
        F.corr("customer_tenure_days", "order_total_spent").alias("corr_tenure_order_total_spent"),
        F.corr("customer_tenure_days", "customer_lifetime_value").alias("corr_tenure_clv")
    )
else:
    print("One of the required columns (customer_tenure_days, order_total_spent, customer_lifetime_value) is all NULL or zero; skipping correlation analysis.")
    

Tenure buckets vs. average spending

In [62]:
    
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_tenure_days") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_total_spent") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id"):
    tenure_buckets_df = dataframes["agg_customers"].withColumn(
        "tenure_bucket",
        F.when(F.col("customer_tenure_days") < 30, "<30d")
         .when(F.col("customer_tenure_days") < 90, "30–89d")
         .when(F.col("customer_tenure_days") < 180, "90–179d")
     .when(F.col("customer_tenure_days") < 365, "180–364d")
     .when(F.col("customer_tenure_days") < 730, "1–2y")
     .otherwise("2y+")
)
    

    analysis["customer_tenure_spend_stats"] = (
        tenure_buckets_df
        .groupBy("tenure_bucket")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("order_total_spent").alias("avg_order_total_spent"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.sum("order_total_spent").alias("total_spent")
        ).fillna({
            "customer_count": 0,
            "avg_order_total_spent": 0,
            "avg_clv": 0,
            "total_spent": 0
        })
        .orderBy("tenure_bucket")
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping tenure spend analysis.")

# Overall CLV summary

In [64]:
if (
    not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_order_value")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id")
):
    analysis["clv_summary"] = (
        dataframes["agg_customers"]
        .agg(
            F.countDistinct("customer_id").alias("customers"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.expr(
                "percentile_approx(customer_lifetime_value, array(0.25, 0.5, 0.75))"
            ).alias("clv_percentiles"),
            F.avg("total_revenue").alias("avg_total_revenue"),
            F.avg("avg_order_value").alias("avg_order_value_overall"),
            F.sum("total_revenue").alias("total_revenue_all_customers"),
        )
        # only fill scalar columns; leave clv_percentiles as-is
        .fillna({
            "customers": 0,
            "avg_clv": 0.0,
            "avg_total_revenue": 0.0,
            "avg_order_value_overall": 0.0,
            "total_revenue_all_customers": 0.0,
        })
    )
else:
    print(
        "One of the required columns in agg_customers is all NULL or zero; skipping CLV summary calculation."
    )

CLV buckets and their spending patterns

In [65]:

if not is_column_all_null_or_zero(dataframes["agg_customers"],"customer_lifetime_value"):
    clv_buckets_df = dataframes["agg_customers"].withColumn(
        "clv_bucket",
        F.when(F.col("customer_lifetime_value") < 100, "<100")
        .when(F.col("customer_lifetime_value") < 500, "100–499")
        .when(F.col("customer_lifetime_value") < 1000, "500–999")
        .when(F.col("customer_lifetime_value") < 5000, "1000–4999")
        .otherwise("5000+")
    )

    analysis["clv_spend_stats"] = (
        clv_buckets_df
        .groupBy("clv_bucket")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("total_revenue").alias("avg_total_revenue"),
            F.avg("avg_order_value").alias("avg_order_value"),
            F.sum("total_revenue").alias("total_revenue_bucket")
        ).fillna({
            "customer_count": 0,
            "avg_clv": 0,
            "avg_total_revenue": 0,
            "avg_order_value": 0,
            "total_revenue_bucket": 0
        })
        .orderBy("clv_bucket")
    )
else:
    print("customer_lifetime_value column is all NULL or zero; skipping CLV bucket statistics calculation.")


Relationship between CLV, total_revenue, and avg_order_value

In [66]:
    
if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_order_value"):
    analysis["clv_correlations"] = dataframes["agg_customers"].select(
        F.corr("customer_lifetime_value", "total_revenue").alias("corr_clv_total_revenue"),
        F.corr("customer_lifetime_value", "avg_order_value").alias("corr_clv_avg_order_value")
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping CLV correlation analysis.")


High-CLV customer segment (top X%)

In [67]:
def high_clv_customers_simple(customers_df, x_percent):
    cutoff_value = customers_df.approxQuantile("customer_lifetime_value", [1 - x_percent/100.0], 0.01)[0]
    return customers_df.filter(F.col("customer_lifetime_value") >= cutoff_value).agg(
        F.countDistinct("customer_id").alias("high_clv_customers"),
        F.avg("customer_lifetime_value").alias("avg_clv"),
        F.avg("avg_order_value").alias("avg_order_value"),
        F.sum("total_revenue").alias("approx_total_revenue")
    ).fillna({
        "high_clv_customers": 0,
        "avg_clv": 0,
        "avg_order_value": 0,
        "approx_total_revenue": 0
    })

try:
    if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_order_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue"):
        analysis["high_clv_customers"] = high_clv_customers_simple(dataframes["agg_customers"], 10) 
        analysis["high_clv_customers"].show()
except Exception as e:
    print(f"Error: {e}")

+------------------+-----------------+-----------------+--------------------+
|high_clv_customers|          avg_clv|  avg_order_value|approx_total_revenue|
+------------------+-----------------+-----------------+--------------------+
|                85|5240.015764705882|1302.903323529412|           445401.34|
+------------------+-----------------+-----------------+--------------------+



Top customers in terms of spending 

Revenue based on customersegemtns and geolocation

In [68]:
def revenue_by_segment(df, group_cols):
    grouped = (
        df.groupBy(*group_cols)
          .agg(
              F.countDistinct("customer_id").alias("customer_count"),
              F.sum("total_revenue").alias("segment_revenue")
          )
    )

    total_rev = grouped.agg(F.sum("segment_revenue").alias("total_revenue_all")).first()[0] or 0.0

    result = (
        grouped
        .withColumn(
            "revenue_per_customer",
            F.when(F.col("customer_count") > 0,
                   F.col("segment_revenue") / F.col("customer_count"))
             .otherwise(F.lit(0.0))
        )
        .withColumn(
            "revenue_share",
            F.when(F.lit(total_rev) > 0,
                   F.col("segment_revenue") / F.lit(total_rev))
             .otherwise(F.lit(0.0))
        )
        .orderBy(F.col("segment_revenue").desc_nulls_last())
    )
    return result

if not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue"):
    analysis["rev_by_country_city"] = revenue_by_segment(dataframes["agg_customers"], ["country", "city"])
    analysis["rev_by_customer_segment"] = revenue_by_segment(dataframes["agg_customers"], ["customer_segment"])
    analysis["rev_by_rfm_segment"] = revenue_by_segment(dataframes["agg_customers"], ["rfm_segment"])
    analysis["rev_by_segment_label"] = revenue_by_segment(dataframes["agg_customers"], ["customer_segment_label"])
    analysis["rev_by_referrer"] = revenue_by_segment(dataframes["agg_customers"], ["preferred_referrer_source"])
    analysis["rev_by_device"] = revenue_by_segment(dataframes["agg_customers"], ["preferred_device_type"])
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping revenue by segment analysis.")

In [69]:
analysis["rev_by_country_city"].show(3)

+-------+----------+--------------+---------------+--------------------+--------------------+
|country|      city|customer_count|segment_revenue|revenue_per_customer|       revenue_share|
+-------+----------+--------------+---------------+--------------------+--------------------+
|   NULL|      NULL|            15|       26654.91|            1776.994|0.017587936747455835|
|    NaN| marieland|             1|        8692.74|             8692.74|0.005735804821028442|
| France|Miltenberg|             1|        8483.09|             8483.09|0.005597469672303344|
+-------+----------+--------------+---------------+--------------------+--------------------+
only showing top 3 rows



Discount based analysis, customers contribution based on level of discount 

In [70]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "total_discount_received") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_discount_per_order") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_orders"):
    disc_df = (
        dataframes["agg_customers"]
        .fillna({
            "total_discount_received": 0.0,
            "total_revenue": 0.0,
            "avg_discount_per_order": 0.0,
            "customer_lifetime_value": 0.0,
            "total_orders": 0
        })
        .withColumn(
            "discount_share_of_revenue",
            F.when(F.col("total_revenue") > 0,
                F.col("total_discount_received") / F.col("total_revenue"))
            .otherwise(F.lit(0.0))
        )
    )
    analysis["discount_customers"] = (
        disc_df
        .withColumn(
            "is_discount_hunter",
            F.when(
                (F.col("discount_share_of_revenue") >= 0.3) &
                (F.col("total_orders") >= 3),
                F.lit(1)
            ).otherwise(F.lit(0))
        )
    )
    analysis["discount_customers_summary"] = (
        analysis["discount_customers"]
        .groupBy("is_discount_hunter")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("discount_share_of_revenue").alias("avg_discount_share"),
            F.avg("avg_discount_per_order").alias("avg_discount_per_order"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("total_revenue").alias("avg_revenue")
        ).fillna({
            "customer_count": 0,
            "avg_discount_share": 0.0,
            "avg_discount_per_order": 0.0,
            "avg_clv": 0.0,
            "avg_revenue": 0.0
        })
    )

    analysis["correlation_discount_vs_clv"] = disc_df.select(
    F.corr("discount_share_of_revenue", "customer_lifetime_value").alias("corr_discount_share_clv"),
    F.corr("avg_discount_per_order", "customer_lifetime_value").alias("corr_avg_discount_clv")
    )

else:
    print("analysis skipped because one of the above columns is missing")

Discount/CLV buckets to see patterns

Bucket by discount_share_of_revenue

In [71]:
if disc_df:
    disc_bucketed = disc_df.withColumn(
        "discount_intensity_bucket",
        F.when(F.col("discount_share_of_revenue") < 0.1, "<10%")
        .when(F.col("discount_share_of_revenue") < 0.25, "10–24%")
        .when(F.col("discount_share_of_revenue") < 0.5, "25–49%")
        .otherwise("50%+")
    )

    analysis["discount_vs_clv"] = (
        disc_bucketed
        .groupBy("discount_intensity_bucket")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("discount_share_of_revenue").alias("avg_discount_share"),
            F.avg("avg_discount_per_order").alias("avg_discount_per_order"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("total_revenue").alias("avg_revenue")
        ).fillna({
            "customer_count": 0,
            "avg_discount_share": 0.0,
            "avg_discount_per_order": 0.0,
            "avg_clv": 0.0,
            "avg_revenue": 0.0
        })
        .orderBy("discount_intensity_bucket")
    )

else:
    print("skipping discount share analysis because disc_df is null")

Discount Behavior & Margin Pressure

In [72]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "total_discount_received") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_discount_per_order") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id"):
    discount_behavior = (
        dataframes["agg_customers"]
        .select(
            "customer_id",
            "total_revenue",
            "total_discount_received",
            (F.col("total_discount_received") / F.col("total_revenue")).alias("discount_to_revenue_ratio"),
            "avg_discount_per_order",
            "customer_lifetime_value"
        )
    )

    analysis["high_discount_customers"] = (
        discount_behavior
        .filter(F.col("discount_to_revenue_ratio") > 0.3)  # threshold example
        .orderBy(F.col("discount_to_revenue_ratio").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping discount behavior analysis.")

 Cart & Checkout Health (Abandonment & Lost Value)

In [73]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "total_carts_created") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_abandoned_carts") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_purchased_carts") and not is_column_all_null_or_zero(dataframes["agg_customers"], "cart_abandonment_rate") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_abandoned_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_time_in_cart_days"):
    analysis["cart_behavior_summary"] = (
        dataframes["agg_customers"]
        .agg(
            F.sum("total_carts_created").alias("total_carts_created"),
            F.sum("total_abandoned_carts").alias("total_abandoned_carts"),
            F.sum("total_purchased_carts").alias("total_purchased_carts"),
            F.avg("cart_abandonment_rate").alias("avg_cart_abandonment_rate"),
            F.sum("total_abandoned_value").alias("total_abandoned_value"),
            F.avg("avg_time_in_cart_days").alias("avg_time_in_cart_days")
        )
    )

    analysis["high_value_abandoners"] = (
        dataframes["agg_customers"]
        .select(
            "customer_id",
            "total_abandoned_carts",
            "total_abandoned_value",
            "cart_abandonment_rate",
            "total_revenue",
            "customer_lifetime_value"
        )
        .filter(F.col("total_abandoned_value") > 0)
        .orderBy(F.col("total_abandoned_value").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping cart behavior analysis.")

One of the required columns in agg_customers is all NULL or zero; skipping cart behavior analysis.


Churn Risk Distribution (Portfolio Health)

In [74]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "churn_risk") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_recency_days"):
    analysis["churn_risk_summary"] = (
        dataframes["agg_customers"]
        .groupBy("churn_risk")
        .agg(
            F.countDistinct("customer_id").alias("num_customers"),
            F.sum("total_revenue").alias("total_revenue"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("order_recency_days").alias("avg_recency_days")
        )
        .orderBy("churn_risk")
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping churn risk analysis.")

One of the required columns in agg_customers is all NULL or zero; skipping churn risk analysis.


High‑CLV Customers at Risk of Churn (Immediate Action List)

In [75]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "churn_risk") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_activity_score") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_recency_days") and not is_column_all_null_or_zero(dataframes["agg_customers"], "days_since_last_purchase") and not is_column_all_null_or_zero(dataframes["agg_customers"], "days_since_last_login") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_segment_label") and not is_column_all_null_or_zero(dataframes["agg_customers"], "rfm_segment") and not is_column_all_null_or_zero(dataframes["agg_customers"], "rfm_category"):
    analysis["high_clv_at_risk"] = (
        dataframes["agg_customers"]
        .select(
            "customer_id",
            "customer_lifetime_value",
            "churn_risk",
            "customer_activity_score",
            "order_recency_days",
            "days_since_last_purchase",
            "days_since_last_login",
            "customer_segment_label",
            "rfm_segment",
            "rfm_category"
        )
        .filter(
            (F.col("churn_risk").isin("medium", "high")) &
            (F.col("customer_lifetime_value") > 0)
        )
        .orderBy(F.col("customer_lifetime_value").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping high CLV at risk analysis.")

One of the required columns in agg_customers is all NULL or zero; skipping high CLV at risk analysis.


RFM Segment Summary (Champions, At Risk, etc.)

In [76]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "rfm_segment") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value") and not is_column_all_null_or_zero(dataframes["agg_customers"], "order_recency_days") and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_orders") and not is_column_all_null_or_zero(dataframes["agg_customers"], "avg_order_value"):
    analysis["rfm_segment_summary"] = (
        dataframes["agg_customers"]
        .groupBy("rfm_segment")
        .agg(
            F.countDistinct("customer_id").alias("num_customers"),
            F.sum("total_revenue").alias("total_revenue"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("order_recency_days").alias("avg_recency_days"),
            F.avg("total_orders").alias("avg_total_orders"),
            F.avg("avg_order_value").alias("avg_aov")
        )
        .orderBy(F.col("total_revenue").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping RFM segment summary analysis.")

One of the required columns in agg_customers is all NULL or zero; skipping RFM segment summary analysis.


High‑Intent Non‑Buyers (Fix Funnel / UX)

In [77]:
if not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue") and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id"):
    analysis["high_intent_non_buyers"] = (
        dataframes["agg_customers"]
        .select(
            "customer_id",
            "total_products_viewed",
        "wishlist_items_count",
        "total_carts_created",
        "total_purchased_carts",
        "cart_abandonment_rate",
        "session_conversion_rate"
    )
    .filter(
        (F.col("total_revenue") == 0) &
        (
            (F.col("total_products_viewed") > 10) |
            (F.col("wishlist_items_count") > 5) |
            (F.col("total_carts_created") > 3)
        )
    )
    .orderBy(F.col("total_products_viewed").desc())
    )
else:
    print("One of the required columns in agg_customers is all NULL or zero; skipping high intent non-buyers analysis.")

# OVERALL CUSTOMER ANALYSIS SUMMARY

Signup cohort: retention by month (based on first order month)

In [117]:

if (
    not is_column_all_null_or_zero(dataframes["agg_customers"], "account_created_at")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "first_order_date")
):
    customers_with_first_order = (
        dataframes["agg_customers"]
        .select(
            "customer_id",
            F.to_date("account_created_at").alias("signup_date"),
            F.to_date("first_order_date").alias("first_order_date")
        )
        .filter(F.col("first_order_date").isNotNull())
    )

    # Define signup cohort month and first-order month
    analysis["customers_cohorts"] = (
        customers_with_first_order
        .withColumn(
            "signup_cohort_month",
            F.date_trunc("month", F.col("signup_date")).cast("date")
        )
        .withColumn(
            "first_order_month",
            F.date_trunc("month", F.col("first_order_date")).cast("date")
        )
    )

    analysis["signup_cohort_summary"] = (
        analysis["customers_cohorts"]
        .groupBy("signup_cohort_month")
        .agg(
            F.countDistinct("customer_id").alias("cohort_customers")
        )
    )
else:
    print("account_created_at or first_order_date is all NULL/zero; skipping signup cohort prep.")

Order‑based monthly cohorts and retention curve

In [118]:

if (
    "agg_orders" in dataframes
    and not is_column_all_null_or_zero(dataframes["agg_orders"], "order_placed_at")
    and not is_column_all_null_or_zero(dataframes["agg_orders"], "customer_id")
    and "customers_cohorts" in locals()
):
    # Orders with order month
    orders_with_month = (
        dataframes["agg_orders"]
        .select(
            "order_id",
            "customer_id",
            F.date_trunc("month", "order_placed_at").cast("date").alias("order_month")
        )
        .filter(F.col("order_month").isNotNull())
    )

    # Join customers to their signup cohort
    orders_with_cohort = (
        orders_with_month
        .join(
            analysis["customers_cohorts"].select("customer_id", "signup_cohort_month"),
            on="customer_id",
            how="inner"
        )
    )

    # Calculate months_since_cohort_start: 0 = cohort month, 1 = next month, etc.
    orders_with_relative_month = (
        orders_with_cohort
        .withColumn(
            "months_since_signup",
            (
                (F.month("order_month") - F.month("signup_cohort_month"))
                + 12 * (F.year("order_month") - F.year("signup_cohort_month"))
            ).cast("int")
        )
        .filter(F.col("months_since_signup") >= 0)
    )

    # Retention: did customer order in each relative month
    analysis["customer_cohort_retention"] = (
        orders_with_relative_month
        .select("signup_cohort_month", "customer_id", "months_since_signup")
        .dropDuplicates(["signup_cohort_month", "customer_id", "months_since_signup"])
        .groupBy("signup_cohort_month", "months_since_signup")
        .agg(
            F.countDistinct("customer_id").alias("active_customers")
        )
        .join(
            analysis["signup_cohort_summary"],
            on="signup_cohort_month",
            how="left"
        )
        .withColumn(
            "retention_rate",
            F.when(
                F.col("cohort_customers") > 0,
                F.col("active_customers") / F.col("cohort_customers")
            ).otherwise(F.lit(0.0))
        )
        .orderBy("signup_cohort_month", "months_since_signup")
    )
else:
    print("Required columns for cohort retention are missing; skipping monthly cohort retention.")

Required columns for cohort retention are missing; skipping monthly cohort retention.


Customer 360 / health table

In [119]:
required_cols_360 = [
    "customer_id",
    "account_created_at",
    "account_status",
    "is_active",
    "is_repeat_customer",
    "total_orders",
    "total_items_purchased",
    "total_cancelled_orders",
    "total_reviews_written",
    "total_sessions",
    "total_pages_viewed",
    "total_products_viewed",
    "wishlist_items_count",
    "total_carts_created",
    "total_abandoned_carts",
    "total_purchased_carts",
    "order_frequency",
    "gender",
    "customer_age_group",
    "city",
    "state_province",
    "country",
    "order_recency_days",
    "customer_tenure_days",
    "days_since_last_login",
    "customer_activity_status",
    "customer_segment",
    "customer_segment_label",
    "rfm_segment",
    "rfm_category",
    "churn_risk",
    "customer_lifetime_value",
    "total_revenue",
    "avg_order_value",
    "avg_items_per_order",
    "total_discount_received",
    "avg_discount_per_order",
    "avg_session_duration",
    "session_conversion_rate",
    "cart_abandonment_rate",
    "preferred_device_type",
    "preferred_referrer_source",
    "preferred_payment_method",
    "wishlist_conversion_rate",
    "days_since_last_purchase",
    "cancellation_rate",
    "customer_activity_score",
    "total_abandoned_value",
    "avg_time_in_cart_days",
    "customer_abandonment_rate",
    "customer_purchase_rate",
]

if "agg_customers" in dataframes:
    missing_for_360 = [c for c in required_cols_360 if c not in dataframes["agg_customers"].columns]
    if missing_for_360:
        print(f"Some 360 columns are missing in agg_customers: {missing_for_360}")
    
    available_cols_360 = [c for c in required_cols_360 if c in dataframes["agg_customers"].columns]
    
    analysis["customer_overall_health_summary"] = dataframes["agg_customers"].select(*available_cols_360)
else:
    print("agg_customers dataframe not found; skipping customer 360 table.")

Systematic cross‑tabs of segmentation fields

 RFM segment × churn_risk

In [120]:
# Cross-tab: RFM segment x churn_risk

if (
    not is_column_all_null_or_zero(dataframes["agg_customers"], "rfm_segment")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "churn_risk")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value")
):
    analysis["rfm_churn_crosstab"] = (
        dataframes["agg_customers"]
        .groupBy("rfm_segment", "churn_risk")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.sum("total_revenue").alias("segment_revenue"),
            F.avg("customer_lifetime_value").alias("avg_clv")
        )
        .orderBy("rfm_segment", "churn_risk")
    )
else:
    print("One of (rfm_segment, churn_risk, customer_id, total_revenue, customer_lifetime_value) is all NULL/zero; skipping rfm x churn analysis.")

One of (rfm_segment, churn_risk, customer_id, total_revenue, customer_lifetime_value) is all NULL/zero; skipping rfm x churn analysis.


In [121]:
# Cross-tab: customer_segment_label x preferred_referrer_source

if (
    not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_segment_label")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "preferred_referrer_source")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue")
):
    analysis["seg_referrer_crosstab"] = (
        dataframes["agg_customers"]
        .groupBy("customer_segment_label", "preferred_referrer_source")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.sum("total_revenue").alias("segment_revenue"),
            F.avg("total_revenue").alias("avg_revenue_per_customer")
        )
        .orderBy("customer_segment_label", F.col("segment_revenue").desc())
    )
else:
    print("One of the required columns is all NULL/zero; skipping segment x referrer analysis.")

One of the required columns is all NULL/zero; skipping segment x referrer analysis.


In [122]:
# Cross-tab: customer_segment_label x preferred_device_type

if (
    not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_segment_label")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "preferred_device_type")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_id")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue")
):
    analysis["seg_device_crosstab"] = (
        dataframes["agg_customers"]
        .groupBy("customer_segment_label", "preferred_device_type")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.sum("total_revenue").alias("segment_revenue"),
            F.avg("total_revenue").alias("avg_revenue_per_customer")
        )
        .orderBy("customer_segment_label", F.col("segment_revenue").desc())
    )
else:
    print("One of the required columns is all NULL/zero; skipping segment x device analysis.")

One of the required columns is all NULL/zero; skipping segment x device analysis.


Payment method vs CLV / churn

In [123]:
# Payment method vs CLV and churn

if (
    not is_column_all_null_or_zero(dataframes["agg_customers"], "preferred_payment_method")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "churn_risk")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue")
):
    analysis["payment_method_vs_clv_churn"] = (
        dataframes["agg_customers"]
        .groupBy("preferred_payment_method", "churn_risk")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("total_revenue").alias("avg_revenue_per_customer"),
            F.sum("total_revenue").alias("total_revenue")
        )
        .orderBy("preferred_payment_method", "churn_risk")
    )

    # Also a simpler view aggregated only by payment method
    analysis["payment_method_summary"] = (
        dataframes["agg_customers"]
        .groupBy("preferred_payment_method")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("total_revenue").alias("avg_revenue_per_customer"),
            F.sum("total_revenue").alias("total_revenue")
        )
        .orderBy(F.col("total_revenue").desc())
    )
else:
    print("One of (preferred_payment_method, CLV, churn_risk, total_revenue) is all NULL/zero; skipping payment vs CLV/churn analysis.")

One of (preferred_payment_method, CLV, churn_risk, total_revenue) is all NULL/zero; skipping payment vs CLV/churn analysis.


 Channel (referrer) vs CLV / churn / discount reliance

In [124]:
# Channel (referrer) vs CLV, churn, and discount reliance

if (
    not is_column_all_null_or_zero(dataframes["agg_customers"], "preferred_referrer_source")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "customer_lifetime_value")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "churn_risk")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_revenue")
    and not is_column_all_null_or_zero(dataframes["agg_customers"], "total_discount_received")
):
    referrer_df = (
        dataframes["agg_customers"]
        .withColumn(
            "discount_share_of_revenue",
            F.when(F.col("total_revenue") > 0,
                   F.col("total_discount_received") / F.col("total_revenue"))
             .otherwise(F.lit(0.0))
        )
    )

    analysis["referrer_source_summary"] = (
        referrer_df
        .groupBy("preferred_referrer_source")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.sum("total_revenue").alias("total_revenue"),
            F.avg("total_revenue").alias("avg_revenue_per_customer"),
            F.avg("discount_share_of_revenue").alias("avg_discount_share")
        ).fillna({
            "customer_count": 0,
            "avg_clv": 0.0,
            "total_revenue": 0.0,
            "avg_revenue_per_customer": 0.0,
            "avg_discount_share": 0.0
        })
        .orderBy(F.col("total_revenue").desc_nulls_last())
    )

    analysis["referrer_churn_summary"] = (
        referrer_df
        .groupBy("preferred_referrer_source", "churn_risk")
        .agg(
            F.countDistinct("customer_id").alias("customer_count"),
            F.avg("customer_lifetime_value").alias("avg_clv"),
            F.avg("discount_share_of_revenue").alias("avg_discount_share")
        ).fillna({
            "customer_count": 0,
            "avg_clv": 0.0,
            "avg_discount_share": 0.0
        })
        .orderBy("preferred_referrer_source", "churn_risk")
    )
else:
    print("One of the required columns for referrer vs CLV/churn/discount is all NULL/zero; skipping analysis.")

One of the required columns for referrer vs CLV/churn/discount is all NULL/zero; skipping analysis.


Profit‑per‑customer derived from agg_orders

In [125]:
# Profit per customer from agg_orders

if (
    "agg_orders" in dataframes
    and not is_column_all_null_or_zero(dataframes["agg_orders"], "customer_id")
    and not is_column_all_null_or_zero(dataframes["agg_orders"], "order_profit")
    and not is_column_all_null_or_zero(dataframes["agg_orders"], "net_profit")
    and not is_column_all_null_or_zero(dataframes["agg_orders"], "order_id")
):
    customer_profit = (
        dataframes["agg_orders"]
        .groupBy("customer_id")
        .agg(
            F.countDistinct("order_id").alias("orders_count_profit"),
            F.sum("order_profit").alias("total_order_profit"),
            F.avg("order_profit").alias("avg_profit_per_order"),
            F.sum("net_profit").alias("total_net_profit"),
            F.avg("net_profit").alias("avg_net_profit_per_order")
        )
    )

    # Join with agg_customers to see CLV vs profit
    if "agg_customers" in dataframes:
        customers_with_profit = (
            dataframes["agg_customers"]
            .join(customer_profit, on="customer_id", how="left")
        )

        # Example aggregated view: CLV vs profit per customer segment
        if (
            "customer_segment_label" in customers_with_profit.columns
            and "total_revenue" in customers_with_profit.columns
            and "customer_lifetime_value" in customers_with_profit.columns
        ):
            analysis["customer_profit_per_segment"] = (
                customers_with_profit
                .groupBy("customer_segment_label")
                .agg(
                    F.countDistinct("customer_id").alias("customer_count"),
                    F.sum("total_revenue").alias("total_revenue"),
                    F.sum("total_order_profit").alias("total_order_profit"),
                    F.sum("total_net_profit").alias("total_net_profit"),
                    F.avg("customer_lifetime_value").alias("avg_clv"),
                    F.avg("total_order_profit").alias("avg_profit_per_customer")
                )
                .orderBy(F.col("total_order_profit").desc_nulls_last())
            )
    else:
        print("agg_customers not found; created customer_profit only.")
else:
    print("One of the required columns in agg_orders is all NULL/zero; skipping customer profit analysis.")

# Product-level analysis

In [ ]:
product_analysis = {}

Overall Best Selling Products

In [ ]:
if (
    "agg_products" in dataframes
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_name")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "category")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "total_units_sold")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "total_orders")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "total_revenue")
):
    product_analysis["best_selling_products"] = (
        dataframes["agg_products"]
        .select(
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "brand",
            "total_units_sold",
            "total_orders",
            "total_revenue",
        )
        .fillna(
            {
                "total_units_sold": 0,
                "total_orders": 0,
                "total_revenue": 0.0,
            }
        )
        .orderBy(
            F.col("total_units_sold").desc_nulls_last(),
            F.col("total_revenue").desc_nulls_last(),
        )
    )
else:
    print(
        "One of the required columns in agg_products is all NULL or zero; "
        "skipping best selling products analysis."
    )

product level seasonal trends analysis

In [ ]:

if (
    not is_column_all_null_or_zero(dataframes["agg_orders"], "order_id")
    and not is_column_all_null_or_zero(dataframes["agg_orders"], "order_placed_at")
    and not is_column_all_null_or_zero(dataframes["agg_order_items"], "order_id")
    and not is_column_all_null_or_zero(dataframes["agg_order_items"], "product_id")
    and not is_column_all_null_or_zero(dataframes["agg_order_items"], "quantity")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_name")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "category")
):
    # Base joined fact
    order_product = (
        dataframes["agg_order_items"]
        .join(
            dataframes["agg_orders"]
            .select("order_id", "order_placed_at"),
            on="order_id",
            how="inner",
        )
        .join(
            dataframes["agg_products"]
            .select("product_id", "product_name", "category", "sub_category", "brand"),
            on="product_id",
            how="left",
        )
        .withColumn("order_date", F.to_date("order_placed_at"))
    )

    # Monthly grain: year + month
    order_product_monthly = add_time_grain(
        order_product, date_col="order_date", grain="month"
    )

    product_analysis["product_monthly_trends"] = (
        order_product_monthly
        .groupBy(
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "brand",
            "grain_year",
            "grain_month",
        )
        .agg(
            F.sum("quantity").alias("units_sold"),
            F.countDistinct("order_id").alias("orders_count"),
        )
        .fillna(
            {
                "units_sold": 0,
                "orders_count": 0,
            }
        )
        .orderBy("product_id", "grain_year", "grain_month")
    )
else:
    print(
        "One of the required columns in agg_orders, agg_order_items, or agg_products is all NULL or zero; "
        "skipping product monthly seasonal trends."
    )

Category-level monthly seasonal trends

In [ ]:
if "order_product" in locals():
    order_product_monthly_cat = add_time_grain(
        order_product, date_col="order_date", grain="month"
    )

    product_analysis["category_monthly_trends"] = (
        order_product_monthly_cat
        .groupBy("category", "grain_year", "grain_month")
        .agg(
            F.sum("quantity").alias("units_sold"),
            F.countDistinct("order_id").alias("orders_count"),
        )
        .fillna(
            {
                "units_sold": 0,
                "orders_count": 0,
            }
        )
        .orderBy("category", "grain_year", "grain_month")
    )
else:
    print("order_product dataset not available; skipping category monthly trends.")

 “Classical” calendar-month seasonality (across years)

In [ ]:
if "order_product" in locals():
    product_analysis["product_calendar_month_seasonality"] = (
        order_product
        .withColumn("calendar_month", F.month("order_date"))
        .groupBy(
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "brand",
            "calendar_month",
        )
        .agg(
            F.sum("quantity").alias("units_sold"),
            F.countDistinct("order_id").alias("orders_count"),
        )
        .fillna(
            {
                "units_sold": 0,
                "orders_count": 0,
            }
        )
        .orderBy("product_id", "calendar_month")
    )

    product_analysis["category_calendar_month_seasonality"] = (
        order_product
        .withColumn("calendar_month", F.month("order_date"))
        .groupBy("category", "calendar_month")
        .agg(
            F.sum("quantity").alias("units_sold"),
            F.countDistinct("order_id").alias("orders_count"),
        )
        .fillna(
            {
                "units_sold": 0,
                "orders_count": 0,
            }
        )
        .orderBy("category", "calendar_month")
    )
else:
    print(
        "order_product dataset not available; skipping calendar-month seasonality analyses."
    )

Products with highest profit margin

In [ ]:
if (
    "agg_products" in dataframes
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_name")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "category")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "profit_margin")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "total_revenue")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "total_units_sold")
):
    product_analysis["highest_margin_products"] = (
        dataframes["agg_products"]
        .select(
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "brand",
            "profit_margin",
            "total_revenue",
            "total_units_sold",
            "total_orders",
        )
        .fillna(
            {
                "profit_margin": 0.0,
                "total_revenue": 0.0,
                "total_units_sold": 0,
                "total_orders": 0,
            }
        )
        .orderBy(
            F.col("profit_margin").desc_nulls_last(),
            F.col("total_revenue").desc_nulls_last(),
        )
    )
else:
    print(
        "One of the required columns in agg_products is all NULL or zero; "
        "skipping highest margin products analysis."
    )

Products with low profit margin but high traffic

In [ ]:
# Products with low margin but high traffic / interest

if (
    "agg_products" in dataframes
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_name")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "category")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "profit_margin")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "total_units_sold")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "total_orders")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "total_revenue")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "view_to_purchase_rate")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "revenue_per_view")
):
    product_analysis["low_margin_high_traffic_products"] = (
        dataframes["agg_products"]
        .select(
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "brand",
            "profit_margin",
            "total_revenue",
            "total_units_sold",
            "total_orders",
            "view_to_purchase_rate",
            "revenue_per_view",
            "total_wishlist_adds",
            "total_cart_adds",
        )
        .fillna(
            {
                "profit_margin": 0.0,
                "total_revenue": 0.0,
                "total_units_sold": 0,
                "total_orders": 0,
                "view_to_purchase_rate": 0.0,
                "revenue_per_view": 0.0,
                "total_wishlist_adds": 0,
                "total_cart_adds": 0,
            }
        )
    )

    # Optional: derive a simple "traffic_score" to help rank
    product_analysis["low_margin_high_traffic_products"] = (
        product_analysis["low_margin_high_traffic_products"]
        .withColumn(
            "traffic_score",
            (
                F.col("total_units_sold")
                + F.col("total_orders")
                + F.col("total_wishlist_adds")
                + F.col("total_cart_adds")
            )
        )
        # Focus on *low* margin but *high* traffic
        # You can tune thresholds; here we don't hard-filter, we just sort to surface likely candidates
        .orderBy(
            F.col("profit_margin").asc_nulls_last(),      # lowest margin first
            F.col("traffic_score").desc_nulls_last(),     # highest traffic next
            F.col("total_revenue").desc_nulls_last(),     # then revenue
        )
    )
else:
    print(
        "One of the required columns in agg_products is all NULL or zero; "
        "skipping low-margin high-traffic products analysis."
    )

Out-of-stock products

In [ ]:
# Out-of-stock products (product-centric view)

if (
    "agg_products" in dataframes
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_name")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "category")
    and ("current_stock_level" in dataframes["agg_products"].columns
         or "current_stock" in dataframes["agg_products"].columns)
):
    products_df = dataframes["agg_products"]

    # Prefer current_stock_level if present, else fall back to current_stock
    stock_col = "current_stock_level" if "current_stock_level" in products_df.columns else "current_stock"

    product_analysis["out_of_stock_products"] = (
        products_df
        .select(
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "brand",
            F.col(stock_col).alias("current_stock_level"),
            "total_units_sold",
            "total_orders",
            "total_revenue",
        )
        .fillna(
            {
                "current_stock_level": 0,
                "total_units_sold": 0,
                "total_orders": 0,
                "total_revenue": 0.0,
            }
        )
        .filter(F.col("current_stock_level") <= 0)
        .orderBy(
            F.col("total_revenue").desc_nulls_last(),
            F.col("total_units_sold").desc_nulls_last(),
        )
    )
else:
    print(
        "One of the required columns in agg_products is all NULL or zero (product_id, product_name, category, stock); "
        "skipping product out-of-stock analysis."
    )

Products with low sell-through rate

In [ ]:
# Products with low view-to-purchase / cart-to-purchase / wishlist-to-purchase rates

if (
    "agg_products" in dataframes
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "product_name")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "category")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "view_to_purchase_rate")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "cart_to_purchase_rate")
    and not is_column_all_null_or_zero(dataframes["agg_products"], "wishlist_to_purchase_rate")
):
    product_analysis["low_conversion_products"] = (
        dataframes["agg_products"]
        .select(
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "brand",
            "view_to_purchase_rate",
            "cart_to_purchase_rate",
            "wishlist_to_purchase_rate",
            "total_units_sold",
            "total_orders",
            "total_wishlist_adds",
            "total_cart_adds",
            "total_revenue",
        )
        .fillna(
            {
                "view_to_purchase_rate": 0.0,
                "cart_to_purchase_rate": 0.0,
                "wishlist_to_purchase_rate": 0.0,
                "total_units_sold": 0,
                "total_orders": 0,
                "total_wishlist_adds": 0,
                "total_cart_adds": 0,
                "total_revenue": 0.0,
            }
        )
        .orderBy(
            F.col("view_to_purchase_rate").asc_nulls_last(),
            F.col("cart_to_purchase_rate").asc_nulls_last(),
            F.col("wishlist_to_purchase_rate").asc_nulls_last(),
            F.col("total_revenue").desc_nulls_last(),
        )
    )
else:
    print(
        "One of the required columns in agg_products is all NULL or zero; "
        "skipping low view/cart/wishlist to purchase rate analysis."
    )

Product rating distribution

In [ ]:
# Product rating distribution

if (
    "agg_reviews" in dataframes
    and not is_column_all_null_or_zero(dataframes["agg_reviews"], "product_id")
    and not is_column_all_null_or_zero(dataframes["agg_reviews"], "rating")
):
    product_analysis["product_rating_distribution"] = (
        dataframes["agg_reviews"]
        .groupBy("product_id", "rating")
        .agg(
            F.count("*").alias("rating_count")
        )
        .fillna({"rating_count": 0})
        .orderBy("product_id", "rating")
    )
else:
    print(
        "One of the required columns in agg_reviews is all NULL or zero; "
        "skipping product rating distribution analysis."
    )

In [ ]:
if (
    "agg_reviews" in dataframes
    and not is_column_all_null_or_zero(dataframes["agg_reviews"], "product_id")
    and not is_column_all_null_or_zero(dataframes["agg_reviews"], "rating")
):
    product_analysis["product_rating_summary"] = (
        dataframes["agg_reviews"]
        .groupBy("product_id")
        .agg(
            F.count("*").alias("total_reviews"),
            F.avg("rating").alias("avg_rating")
        )
        .fillna({"total_reviews": 0, "avg_rating": 0.0})
        .orderBy(F.col("avg_rating").desc_nulls_last())
    )

Product/category viewing patterns

Category-level viewing effectiveness

In [126]:
if not is_column_all_null_or_zero(dataframes["agg_products"], "category") and not is_column_all_null_or_zero(dataframes["agg_products"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_units_sold") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_orders") and not is_column_all_null_or_zero(dataframes["agg_products"], "view_to_purchase_rate") and not is_column_all_null_or_zero(dataframes["agg_products"], "revenue_per_view") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_revenue"):
    analysis["category_view_patterns"] = (
        dataframes["agg_products"]
        .groupBy("category")
        .agg(
            F.countDistinct("product_id").alias("products_in_category"),
            F.sum("total_units_sold").alias("total_units_sold"),
            F.sum("total_orders").alias("total_orders"),
            F.avg("view_to_purchase_rate").alias("avg_view_to_purchase_rate"),
            F.avg("revenue_per_view").alias("avg_revenue_per_view"),
            F.sum("total_revenue").alias("total_revenue")
        ).fillna({
            "products_in_category": 0,
            "total_units_sold": 0,
            "total_orders": 0,
            "avg_view_to_purchase_rate": 0.0,
            "avg_revenue_per_view": 0.0,
            "total_revenue": 0.0
        })
        .orderBy(F.col("total_revenue").desc_nulls_last())
    )
else:
    print("One of the required columns in agg_products is all NULL or zero; skipping category view patterns analysis.")

One of the required columns in agg_products is all NULL or zero; skipping category view patterns analysis.


Product-level Top-View-to-Purchase Rates

In [127]:
if not is_column_all_null_or_zero(dataframes["agg_products"], "view_to_purchase_rate") and not is_column_all_null_or_zero(dataframes["agg_products"], "revenue_per_view") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_units_sold") and not is_column_all_null_or_zero(dataframes["agg_products"], "total_orders"):
    analysis["top_view_to_purchase_products"] = (
        dataframes["agg_products"]
        .select(
            "product_id",
            "product_name",
            "category",
            "view_to_purchase_rate",
            "revenue_per_view",
            "total_units_sold",
            "total_orders"
        ).fillna({
            "view_to_purchase_rate": 0.0,
            "revenue_per_view": 0.0,
            "total_units_sold": 0,
            "total_orders": 0
        })
        .orderBy(F.col("view_to_purchase_rate").desc_nulls_last())
    )
else:
    print("One of the required columns in agg_products is all NULL or zero; skipping top view to purchase products analysis.")

One of the required columns in agg_products is all NULL or zero; skipping top view to purchase products analysis.


# Wishlist usage and conversion rate

Overall wishlist usage and conversion

In [128]:
if not is_column_all_null_or_zero(dataframes["agg_wishlist"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_wishlist"], "purchased_date") and not is_column_all_null_or_zero(dataframes["agg_wishlist"], "customer_id"):
    analysis["wishlist_overall_summary"] = dataframes["agg_wishlist"].agg(
        F.count("*").alias("total_wishlist_items"),
        F.countDistinct("customer_id").alias("customers_using_wishlist"),
        F.countDistinct("product_id").alias("products_in_wishlist"),
        F.sum(F.when(F.col("purchased_date").isNotNull(), 1).otherwise(0)).alias("wishlist_purchased_items")
    ).withColumn(
        "wishlist_conversion_rate",
        F.col("wishlist_purchased_items") / F.col("total_wishlist_items")
    ).fillna({
        "total_wishlist_items": 0,
        "customers_using_wishlist": 0,
        "products_in_wishlist": 0,
        "wishlist_purchased_items": 0,
        "wishlist_conversion_rate": 0.0
    })
else:
    print("One of the required columns in agg_wishlist is all NULL or zero; skipping wishlist overall analysis.")

Wishlist usage & conversion by product

In [129]:
if not is_column_all_null_or_zero(dataframes["agg_wishlist"], "product_id") and not is_column_all_null_or_zero(dataframes["agg_wishlist"], "purchased_date"):
    analysis["wishlist_by_product"] = (
        dataframes["agg_wishlist"]  
        .groupBy("product_id")
        .agg(
            F.count("*").alias("wishlist_adds"),
            F.sum(F.when(F.col("purchased_date").isNotNull(), 1).otherwise(0)).alias("wishlist_purchases")
        )
        .withColumn(
            "wishlist_conversion_rate",
            F.col("wishlist_purchases") / F.col("wishlist_adds")
        )
    )
else:
    print("One of the required columns in agg_wishlist is all NULL or zero; skipping wishlist by product analysis.")

Wishlist usage & conversion by customer

In [130]:
if not is_column_all_null_or_zero(dataframes["agg_wishlist"], "customer_id") and not is_column_all_null_or_zero(dataframes["agg_wishlist"], "purchased_date"):
    analysis["wishlist_by_customer"] = (
        dataframes["agg_wishlist"] 
        .groupBy("customer_id")
        .agg(
            F.count("*").alias("wishlist_adds"),
            F.sum(F.when(F.col("purchased_date").isNotNull(), 1).otherwise(0)).alias("wishlist_purchases")
        )
        .withColumn(
            "wishlist_conversion_rate",
            F.when(F.col("wishlist_adds") > 0,
                F.col("wishlist_purchases") / F.col("wishlist_adds"))
            .otherwise(F.lit(0.0))
        )
    )
else:
    print("customer_id or purchased_date column is all NULL or zero; skipping wishlist by customer analysis.")

# Cart creation, abandonment, and recovery statistics

Basic cart creation & status distribution

In [131]:
if not is_column_all_null_or_zero(dataframes["agg_shopping_cart"], "cart_id") and not is_column_all_null_or_zero(dataframes["agg_shopping_cart"], "cart_status"):
    analysis["cart_overall_stats"] = dataframes["agg_shopping_cart"].agg(
        F.countDistinct("cart_id").alias("total_carts"),
        F.count("*").alias("total_cart_lines")
    ).fillna({
        "total_carts": 0,
        "total_cart_lines": 0
    })
else:
    print("cart_id or cart_status column is all NULL or zero; skipping overall cart statistics analysis.")
if not is_column_all_null_or_zero(dataframes["agg_shopping_cart"], "cart_status"):
    analysis["cart_status_distribution"] = (
        dataframes["agg_shopping_cart"]
        .groupBy("cart_status")
        .agg(
            F.countDistinct("cart_id").alias("carts_count"),
        F.count("*").alias("cart_lines_count")
    ).fillna({
        "carts_count": 0,
        "cart_lines_count": 0
    })
    .orderBy("cart_status")
)
else:
    print("cart_status column is all NULL or zero; skipping cart status distribution analysis.")

Abandonment & recovery (using agg_cart_abandonment_analysis)

In [132]:
dataframes["agg_cart_abandonment_analysis"].show(5)

+-------+-----------+---------------+-----------+----------------+-----------------+-----------------+------------------+------------------------+------------------+-------------------+-----------+-----------------------+----------------+---------------+----------+-------------------+-----------------------+---------------+------------------+----------------------+
|cart_id|cart_status|cart_added_date|customer_id|cart_items_count|session_converted|time_in_cart_days|time_in_cart_hours|recovery_potential_score|  cart_total_value|cart_avg_item_price|device_used|abandoned_cart_category|first_added_date|last_added_date|session_id|cart_status_derived|cart_abandonment_reason|cart_value_tier|cart_size_category|abandonment_risk_score|
+-------+-----------+---------------+-----------+----------------+-----------------+-----------------+------------------+------------------------+------------------+-------------------+-----------+-----------------------+----------------+---------------+----------

In [133]:
if not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_id") and not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_status"):
    analysis["cart_abandon_summary"] = dataframes["agg_cart_abandonment_analysis"].agg(
        F.countDistinct("cart_id").alias("total_carts_tracked"),
        F.countDistinct(F.when(F.col("cart_status") == "Abandoned", F.col("cart_id"))).alias("abandoned_carts"),
        F.countDistinct(F.when(F.col("cart_status") == "Converted", F.col("cart_id"))).alias("converted_carts")
    ).withColumn(
        "abandonment_rate",
        F.col("abandoned_carts") / F.col("total_carts_tracked")
    ).withColumn(
        "purchase_rate",
        F.col("converted_carts") / F.col("total_carts_tracked")
    ).fillna({
        "total_carts_tracked": 0,
        "abandoned_carts": 0,
        "converted_carts": 0,
        "abandonment_rate": 0.0,
        "purchase_rate": 0.0
    })
else:
    print("cart_id or cart_status column is all NULL or zero; skipping cart abandonment overall analysis.")

In [134]:
analysis["cart_abandon_summary"].show()

+-------------------+---------------+---------------+-----------------+------------------+
|total_carts_tracked|abandoned_carts|converted_carts| abandonment_rate|     purchase_rate|
+-------------------+---------------+---------------+-----------------+------------------+
|               1585|           1003|            441|0.632807570977918|0.2782334384858044|
+-------------------+---------------+---------------+-----------------+------------------+



Value and size characteristics of abandoned vs purchased carts

In [135]:
if not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_status"):
    analysis["cart_value_stats"] = (
        dataframes["agg_cart_abandonment_analysis"]
        .groupBy("cart_status")
        .agg(
            F.countDistinct("cart_id").alias("carts_count"),
            F.avg("cart_total_value").alias("avg_cart_value"),
        F.avg("cart_items_count").alias("avg_cart_items"),
        F.avg("time_in_cart_days").alias("avg_time_in_cart_days"),
        F.avg("recovery_potential_score").alias("avg_recovery_potential_score")
    ).fillna({
        "carts_count": 0,
        "avg_cart_value": 0.0,
        "avg_cart_items": 0.0,
        "avg_time_in_cart_days": 0.0,
        "avg_recovery_potential_score": 0.0
    })
    .orderBy("cart_status")
)
else:
    print("cart_status column is all NULL or zero; skipping cart value statistics analysis.")

Recovery opportunity: high-value abandoned carts

In [136]:
if not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_status") and not is_column_all_null_or_zero(dataframes["agg_cart_abandonment_analysis"], "cart_total_value"):
    analysis["high_value_abandoned_carts"] = (
        dataframes["agg_cart_abandonment_analysis"]
        .filter(
            (F.col("cart_status") == "abandoned") &
            (F.col("cart_total_value") >= 100)  # threshold – adjust as needed
        )
        .select(
            "cart_id",
            "customer_id",
            "cart_total_value",
            "cart_items_count",
            "time_in_cart_days",
            "recovery_potential_score",
            "abandonment_risk_score"
        )
    )
else:
    print("cart_status or cart_total_value column is all NULL or zero; skipping high-value abandoned carts analysis.")

In [142]:
count = 0
for key in analysis.keys():
    count+=1
print(count)
print("Damn Thats a lot of analyses!🧑🏿")

65
Damn Thats a lot of analyses!🧑🏿


In [143]:
for keys in analysis.keys():
    analysis[keys].show(1)


+-------------------+------------+----------------+-------------+------------+----------+-------+----------+
|         grain_date|total_orders|total_units_sold|total_revenue|gross_profit|net_profit|    aov|margin_pct|
+-------------------+------------+----------------+-------------+------------+----------+-------+----------+
|2023-11-18 01:04:47|           1|               0|      1990.81|         0.0|       0.0|1990.81|       0.0|
+-------------------+------------+----------------+-------------+------------+----------+-------+----------+
only showing top 1 row

+----------+----------+------------+----------------+-------------+------------+----------+-----------------+-------------------+
|grain_year|grain_week|total_orders|total_units_sold|total_revenue|gross_profit|net_profit|              aov|         margin_pct|
+----------+----------+------------+----------------+-------------+------------+----------+-----------------+-------------------+
|      2023|        46|           3|     